## 1. Dataset

In [5]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer


In [2]:
# Load dataset
url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
df = pd.read_csv(url, sep='\t', header=None, names=['label', 'message'])

In [3]:
# Encode labels: spam=1, ham=0
df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})


In [4]:
print(df.head())

  label                                            message  label_num
0   ham  Go until jurong point, crazy.. Available only ...          0
1   ham                      Ok lar... Joking wif u oni...          0
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...          1
3   ham  U dun say so early hor... U c already then say...          0
4   ham  Nah I don't think he goes to usf, he lives aro...          0


## 2. Preprocessing & Feature Extraction

In [6]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)  # remove punctuation
    return text

df['clean_message'] = df['message'].apply(clean_text)

# TF-IDF vectorizer (unigrams + bigrams)
vectorizer = TfidfVectorizer(ngram_range=(1,2), max_features=5000)
X = vectorizer.fit_transform(df['clean_message'])
y = df['label_num']

## 3. Train / Test Split

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 4. Model Training

In [8]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model = MultinomialNB()
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)

# Evaluation
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred, target_names=['ham', 'spam']))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.9776
              precision    recall  f1-score   support

         ham       0.97      1.00      0.99       966
        spam       1.00      0.83      0.91       149

    accuracy                           0.98      1115
   macro avg       0.99      0.92      0.95      1115
weighted avg       0.98      0.98      0.98      1115

Confusion Matrix:
 [[966   0]
 [ 25 124]]


## 5. Testing on New Emails

In [9]:
def predict_spam(email_text):
    cleaned = clean_text(email_text)
    vec = vectorizer.transform([cleaned])
    pred = model.predict(vec)[0]
    prob = model.predict_proba(vec)[0][1]  # probability of spam
    return "SPAM" if pred == 1 else "HAM", prob

# Example
emails = [
    "Congratulations! You've won a free iPhone. Click here to claim now.",
    "Hey, are we still meeting for lunch tomorrow?",
    "URGENT: Your account will be closed. Verify your details immediately."
]

for email in emails:
    label, prob = predict_spam(email)
    print(f"Message: {email[:50]}... -> {label} (spam prob: {prob:.3f})")

Message: Congratulations! You've won a free iPhone. Click h... -> SPAM (spam prob: 0.896)
Message: Hey, are we still meeting for lunch tomorrow?... -> HAM (spam prob: 0.004)
Message: URGENT: Your account will be closed. Verify your d... -> HAM (spam prob: 0.486)


## 6. Saving the Model (for reuse)

In [10]:
import joblib

joblib.dump(model, 'spam_model.pkl')
joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')

['tfidf_vectorizer.pkl']